# Notebook 04: Synthetic Task Generator

**Purpose**: Build, validate, and freeze the synthetic task generator. This is SATA's training data source and the testbed for RQ2/RQ4.

## Why synthetic tasks exist at all — the ground-truth problem

Every notebook up to this point works with real TableShift data, where we can measure accuracy but never know for certain *which features the label actually depends on* — a feature's true causal role in, say, income prediction isn't something any dataset can hand you directly. That's a hard ceiling on what real data can support: **π_true (the true feature-importance ranking) simply doesn't exist for TableShift datasets.** Without it, RQ4's success criterion — "does SATA's learned reweighting produce more *correct* reliance, not just higher accuracy?" — has nothing to check against. `src/evaluation/faithfulness_correctness.py::rank_by_true_importance` needs a ground truth to rank against, and only a generator we built ourselves can supply one.

This is also **why SATA has to be meta-trained on synthetic data rather than TableShift itself** (the spec's "data contamination guarantee"): SATA's training signal (`src/models/sata_targets.py::compute_target_scores`) is built directly from ground-truth regime/counter-spurious labels that only exist because we generated the data and know the causal rule. Training SATA on real TableShift rows would mean either fabricating that supervision (defeating the purpose) or training on accuracy alone (which is exactly the faithfulness gap RQ3 exists to expose). Keeping the two arms strictly separate is what lets a later claim like "SATA transfers to real data" (the Week-8 stretch goal referenced in Notebook 03) mean something — the model genuinely never saw TableShift rows during training.

The rule families (linear / threshold / tree / sparse-interaction) and six environments below are exactly the meta-training distribution the lit review describes for SATA (Section 3, Task 2): "meta-trained on a synthetic distribution of tabular tasks spanning linear, threshold, tree-style, and sparse-interaction rules, each instantiated under six shift types."

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

## Generator specification

See `src/data/generator.py::SyntheticTask`. Rule families: linear, threshold, tree (depth 2-3), sparse_interaction. Six environments per task: id, covariate, spurious_reversal, extrapolation, missing_feature, mechanism.

**Why four rule families, and why hold one out entirely?** A generator with only one rule family (say, linear) would let SATA learn a shortcut of its own — "always weight demos this specific way" — that happens to work because every training task shares the same functional form. Four structurally distinct families (linear combination, single/multi threshold, tree-structured regimes, multiplicative interaction) force SATA to learn something about *demonstration relevance relative to a task's regime structure* rather than memorising one rule shape. `sparse_interaction` is held out entirely from meta-training (`heldout_family` in `configs/default.yaml`) specifically so RQ4's accuracy comparison can be run on tasks whose *rule family* SATA has never seen — this is the direct analogue of Goddard et al.'s (2025) pretraining-task-diversity phase transition (Lit-review §2.3.1): below a diversity threshold, ICL-like systems produce solutions that only work on tasks resembling training; above it, they generalise across the task space. Testing on the held-out family is how this project checks which side of that threshold SATA's meta-training lands on.

**Why six environments, specifically these six?** Each is a controlled instantiation of one shift concept from the taxonomy, deliberately built so ground truth is known by construction (no DISDE decomposition needed, unlike TableShift's naturally-occurring shifts): `covariate` moves `P(x)` only; `spurious_reversal` and `mechanism` move `P(y|x)` (concept shift) via two different mechanisms — flipping which value of the spurious feature correlates with the label, vs. changing the causal rule's own coefficients/threshold; `missing_feature` and `extrapolation` are stress tests of robustness rather than points on the covariate/concept axis. `id` is the baseline every other environment is compared against.

In [2]:
from src.data.generator import SyntheticTask, generate_task_suite, RULE_FAMILIES, ENVIRONMENTS

task = SyntheticTask(
    task_id='smoke_test', rule_family='linear', causal_features=[0, 2, 4],
    coefficients=[1.0, -0.5, 2.0], spurious_strength=0.85,
)
X, y, metadata = task.generate_environment('id', n_samples=96, seed=0)
X.shape, y.mean()

((96, 10), np.float64(0.4375))

## Task sampling

`generate_task_suite(config.generator)` samples `n_train_tasks` tasks (excluding the held-out family) plus `n_heldout_family_tasks` tasks of `heldout_family` only.

Four disjoint task pools exist for the same reason train/val/test splits exist anywhere: `tasks_train` is what SATA's weights are fit to (Notebook 05); `tasks_val` is what Gate 2 and early-stopping-style model selection use, so SATA isn't validated on the exact tasks it was trained on; `tasks_test` is the untouched set RQ2's protocol × shift-type grid and most of RQ4's accuracy comparison are computed on; `tasks_heldout_family` is the *sparse_interaction*-only pool used specifically to test generalisation to a rule family SATA never saw a single example of during training.

In [3]:
from src.data.generator import generate_val_test_tasks

tasks = generate_task_suite(config.generator)
train_tasks = [t for t in tasks if t.task_id.startswith('train_')]
heldout_family_tasks = [t for t in tasks if t.task_id.startswith('heldout_')]
val_tasks, test_tasks = generate_val_test_tasks(config.generator)

len(train_tasks), len(val_tasks), len(test_tasks), len(heldout_family_tasks)

(2000, 200, 200, 50)

## XGBoost validation gate (Week 2-3)

For ~50 randomly sampled tasks: fit `XGBClassifier(max_depth=4, n_estimators=100)` on the `id` environment's 64 demos, evaluate on 32 queries from each of the 6 environments. Check:
- ID accuracy > 80%
- Spurious-reversal accuracy < 60%
- Mechanism-shift accuracy < ID accuracy by >=15 points

**Gate criterion**: >=80% of sampled tasks show the expected degradation profile. Only then freeze the generator.

**What this gate is actually checking, and why it's not a formality.** The whole premise of this project — real and synthetic arms alike — is that models exploit shortcuts: features that are predictive in training but not causally load-bearing (Geirhos et al. 2020, Lit-review §2.2.1). If our synthetic tasks *don't* actually exhibit that failure mode — if a simple classifier trained on 64 demos already generalises fine across every environment — then the generator hasn't built a testbed for shortcut learning at all, it's built a testbed for something else, and every downstream number (SATA's Gate 2, RQ2's grid, RQ4's comparison) would be measuring performance on tasks that don't have the property the whole thesis is about.

This gate is a cheap, LLM-free proxy for exactly that check, using XGBoost as a stand-in learner: ID accuracy confirms the task is learnable at all; spurious-reversal accuracy dropping below chance-adjacent levels confirms the classifier actually latched onto the spurious feature rather than the causal one; the mechanism-shift drop confirms concept shift is real, not just relabelling. Nagarajan et al.'s (2021) two-mechanism account of shortcut learning (Lit-review §2.2.1) — a geometric skew favouring low-norm spurious separators, and a statistical skew from gradient descent's convergence rate along the spurious direction — describes *why* a learner would end up here; this gate is the empirical test of whether our generator actually reproduces that pathology, not just assumed it does.

In [4]:
from xgboost import XGBClassifier
import random
import numpy as np
import pandas as pd

def validate_task(task, seed=0):
    demo_X, demo_y, _ = task.generate_environment('id', n_samples=64, seed=seed)
    clf = XGBClassifier(max_depth=4, n_estimators=100, verbosity=0)
    clf.fit(demo_X, demo_y)
    accs = {}
    for env in ENVIRONMENTS:
        qX, qy, _ = task.generate_environment(env, n_samples=32, seed=seed + 1)
        accs[env] = (clf.predict(qX) == qy).mean()
    return accs


sample_rng = random.Random(config.seed_accuracy[0])
sampled_tasks = sample_rng.sample(train_tasks, min(50, len(train_tasks)))

validation_rows = []
for task in sampled_tasks:
    accs = validate_task(task, seed=config.seed_accuracy[0])
    passes_gate = (
        accs['id'] > 0.80
        and accs['spurious_reversal'] < 0.60
        and (accs['id'] - accs['mechanism']) >= 0.15
    )
    validation_rows.append({
        'task_id': task.task_id,
        'rule_family': task.rule_family,
        **{f'acc_{env}': accs[env] for env in ENVIRONMENTS},
        'passes_gate': passes_gate,
    })

validation_df = pd.DataFrame(validation_rows)
pass_rate = float(validation_df['passes_gate'].mean())
gate_passed = pass_rate >= 0.80

print(f"Gate pass rate: {pass_rate:.1%} ({int(validation_df['passes_gate'].sum())}/{len(validation_df)})")
print("GATE PASSED — freezing generator." if gate_passed
      else "GATE FAILED — adjust generator parameters (spurious_strength_range, "
           "coefficient scale, label_noise) and re-run this notebook.")

validation_df

Gate pass rate: 8.0% (4/50)
GATE FAILED — adjust generator parameters (spurious_strength_range, coefficient scale, label_noise) and re-run this notebook.


,task_id,rule_family,acc_id,acc_covariate,acc_spurious_reversal,acc_extrapolation,acc_missing_feature,acc_mechanism,passes_gate
0,train_1309,threshold,0.84375,0.78125,0.62500,0.81250,0.78125,0.84375,False
1,train_0228,linear,0.87500,0.84375,0.34375,0.87500,0.87500,0.93750,False
2,train_0051,tree,0.93750,0.87500,0.81250,0.90625,0.84375,0.90625,False
3,train_1518,tree,0.87500,0.87500,0.56250,0.87500,0.90625,0.75000,False
4,train_0563,tree,0.90625,0.87500,0.93750,0.90625,0.81250,0.59375,False
5,train_0501,tree,0.84375,0.84375,0.84375,0.87500,0.65625,0.84375,False
6,train_0457,tree,0.90625,0.81250,0.46875,0.87500,0.84375,0.90625,False
7,train_0285,linear,0.65625,0.53125,0.50000,0.75000,0.68750,0.68750,False
8,train_1508,threshold,0.84375,0.84375,0.78125,0.75000,0.40625,0.62500,False
9,train_0209,linear,0.96875,0.84375,0.87500,0.81250,0.87500,0.87500,False


In [5]:
import json


def save_task_environments(task, out_dir, n_demos, n_queries, seed):
    """One parquet per task: all 6 environments' demo+query rows, tagged by
    `environment`/`split` columns, plus a small metadata JSON sidecar."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for env in ENVIRONMENTS:
        X, y, metadata = task.generate_environment(env, n_samples=n_demos + n_queries, seed=seed)
        for i in range(n_demos + n_queries):
            row = {f'feature_{j}': X[i, j] for j in range(X.shape[1])}
            row.update({
                'environment': env,
                'split': 'demo' if i < n_demos else 'query',
                'label': metadata[i]['label'],
                'regime': metadata[i]['regime'],
                'is_counter_spurious': metadata[i]['is_counter_spurious'],
                'spurious_consistent': metadata[i]['spurious_consistent'],
            })
            rows.append(row)
    pd.DataFrame(rows).to_parquet(out_dir / f'{task.task_id}.parquet', index=False)

    meta = {
        'task_id': task.task_id,
        'rule_family': task.rule_family,
        'causal_features': list(int(c) for c in task.causal_features),
        'coefficients': [float(c) for c in task.coefficients],
        'spurious_strength': float(task.spurious_strength),
        'threshold': float(task.threshold),
    }
    with open(out_dir / f'{task.task_id}_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)


DATA_SYN = resolve_path(config.paths.data_synthetic)
TASK_GROUPS = {
    'tasks_train': train_tasks,
    'tasks_val': val_tasks,
    'tasks_test': test_tasks,
    'tasks_heldout_family': heldout_family_tasks,
}

# Save regardless of gate outcome — Notebook 05/06 need *something* to develop
# and smoke-test against, and a failed gate is a signal to keep calibrating,
# not a reason to block having any data on disk. `generator_config.json`'s
# `gate_pass_rate`/`frozen` fields make the provisional status explicit; treat
# this as genuinely frozen only once `frozen` is true.
for group_name, group_tasks in TASK_GROUPS.items():
    group_dir = DATA_SYN / group_name
    for task in group_tasks:
        save_task_environments(
            task, group_dir,
            n_demos=config.generator.demos_per_task,
            n_queries=config.generator.queries_per_env,
            seed=config.seed_accuracy[0],
        )
    print(f"{group_name}: saved {len(group_tasks)} tasks -> {group_dir}")

generator_config = {
    'n_features': config.generator.n_features,
    'n_causal_range': list(config.generator.n_causal_range),
    'rule_families': list(config.generator.rule_families),
    'heldout_family': config.generator.heldout_family,
    'spurious_strength_range': list(config.generator.spurious_strength_range),
    'label_noise': config.generator.label_noise,
    'demos_per_task': config.generator.demos_per_task,
    'queries_per_env': config.generator.queries_per_env,
    'environments': ENVIRONMENTS,
    'gate_pass_rate': pass_rate,
    'frozen': gate_passed,
}
with open(DATA_SYN / 'generator_config.json', 'w') as f:
    json.dump(generator_config, f, indent=2)

validation_df.to_parquet(DATA_SYN / 'xgboost_validation.parquet', index=False)

if gate_passed:
    print("Generator frozen — all downstream notebooks should treat these tasks as fixed.")
else:
    print(
        f"Gate NOT passed ({pass_rate:.0%} < 80%) — data saved for development/smoke-testing "
        "Notebooks 05/06 against, but generator_config.json['frozen'] is False. Keep calibrating "
        "spurious_strength_range/label_noise/coefficient scale (see the failure breakdown above — "
        "spurious_reversal accuracy is the main blocker: a 100-estimator XGBoost on 64 demos is "
        "robust enough that even a 0.92-0.99 spurious_strength doesn't reliably mislead it) before "
        "treating results from Notebooks 05+ as final."
    )

tasks_train: saved 2000 tasks -> /Users/chenuka/Documents/USYD/thesis/sata-project/data/synthetic/tasks_train


tasks_val: saved 200 tasks -> /Users/chenuka/Documents/USYD/thesis/sata-project/data/synthetic/tasks_val


tasks_test: saved 200 tasks -> /Users/chenuka/Documents/USYD/thesis/sata-project/data/synthetic/tasks_test
tasks_heldout_family: saved 50 tasks -> /Users/chenuka/Documents/USYD/thesis/sata-project/data/synthetic/tasks_heldout_family
Gate NOT passed (8% < 80%) — data saved for development/smoke-testing Notebooks 05/06 against, but generator_config.json['frozen'] is False. Keep calibrating spurious_strength_range/label_noise/coefficient scale (see the failure breakdown above — spurious_reversal accuracy is the main blocker: a 100-estimator XGBoost on 64 demos is robust enough that even a 0.92-0.99 spurious_strength doesn't reliably mislead it) before treating results from Notebooks 05+ as final.


## Output

- `data/synthetic/tasks_train/` — 2000 task directories
- `data/synthetic/tasks_val/` — 200 tasks
- `data/synthetic/tasks_test/` — 200 tasks
- `data/synthetic/tasks_heldout_family/` — 50 tasks
- `data/synthetic/generator_config.json` — frozen generator parameters
- `data/synthetic/xgboost_validation.parquet` — gate results